### Purpose
Fraud rate, loss exposure, and temporal / ticket-size patterns.

**Data:** `marts.fct_transactions` + dims (Postgres). Requires `make docker-up` + `dbt build`.

**Charts:** `notebooks/charts/fraud_risk/`

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(BASE_DIR))

from notebooks.utils.data_loader import get_merged_data, print_data_sanity
from notebooks.utils.style import C, INR_CR, apply_theme, style_axes
from notebooks.utils.visualization import minimal_heatmap, save_chart, new_fig

apply_theme()
CHART_SUBDIR = "fraud_risk"

df, data_source = get_merged_data()
df["fraud_amount"] = df["amount"].where(df["is_fraud"], 0)
print_data_sanity(df, data_source)

### Chart - Monthly fraud rate

In [ ]:
m = df.groupby("month").agg(fraud_rate=("is_fraud", "mean")).reset_index()
platform = df["is_fraud"].mean() * 100

fig, ax = new_fig(11, 4)
ax.plot(m["month"], m["fraud_rate"] * 100, color=C["danger"], marker="o", markersize=4, linewidth=1.8)
ax.axhline(platform, color=C["neutral"], linestyle="--", linewidth=1.2, label=f"Platform avg {platform:.2f}%")
ax.set_ylabel("Fraud rate (%)")
ax.legend(loc="upper right", fontsize=8)
ax.set_title("Monthly fraud rate", loc="left", fontweight="600")
plt.setp(ax.get_xticklabels(), rotation=0)
style_axes(ax)
save_chart(fig, "monthly_fraud_rate", CHART_SUBDIR)

`Rate is stable ~3.5% — risk teams still watch month-on-month drift even when flat.`

### Chart - Monthly fraud loss (₹)

In [ ]:
loss = df.groupby("month")["fraud_amount"].sum().reset_index(name="fraud_loss_inr")

fig, ax = new_fig(11, 4)
bars = ax.bar(loss["month"], loss["fraud_loss_inr"], color=C["danger"], width=0.65, alpha=0.9)
ax.set_ylabel("Fraud loss (INR)")
ax.yaxis.set_major_formatter(INR_CR)
ax.set_title("Monthly fraud loss", loc="left", fontweight="600")
plt.setp(ax.get_xticklabels(), rotation=0)
style_axes(ax)
save_chart(fig, "monthly_fraud_loss_inr", CHART_SUBDIR)

`Flat fraud % but rising ₹ loss — volume growth increases exposure even when rate is unchanged.`

### Chart - Fraud rate: category × payment method

In [ ]:
pivot = df.pivot_table(index="merchant_category", columns="payment_method", values="is_fraud", aggfunc="mean").mul(100)
vmin, vmax = pivot.min().min() - 0.2, pivot.max().max() + 0.2

fig, ax = new_fig(10, 4.5)
minimal_heatmap(pivot, ax, vmin=vmin, vmax=vmax, cbar_label="Fraud %")
ax.set_title("Fraud rate by category & payment method", loc="left", fontweight="600")
ax.set_xlabel("")
ax.set_ylabel("")
save_chart(fig, "fraud_heatmap_category_payment", CHART_SUBDIR)

`Electronics & Travel run ~4.5–4.9% vs Fashion ~2% — tighten limits on the dark-blue cells.`

### Chart - Fraud exposure by ticket size

In [ ]:
bins = [0, 1000, 5000, 10000, 25000, 50000, float("inf")]
labels = ["<₹1K", "₹1–5K", "₹5–10K", "₹10–25K", "₹25–50K", "₹50K+"]
df["amount_bucket"] = pd.cut(df["amount"], bins=bins, labels=labels, right=False)
stats = df.groupby("amount_bucket", observed=True).agg(
    txns=("transaction_id", "count"),
    fraud_txns=("is_fraud", "sum"),
    fraud_loss=("fraud_amount", "sum"),
).reset_index()
stats["fraud_rate"] = stats["fraud_txns"] / stats["txns"] * 100
plot_stats = stats[stats["fraud_loss"] > 0].copy()

fig, ax = new_fig(7, 4)
bars = ax.barh(
    plot_stats["amount_bucket"].astype(str), plot_stats["fraud_loss"].to_numpy(),
    color=C["danger"], height=0.55, alpha=0.9,
)
ax.set_xlabel("Fraud loss (INR)")
ax.xaxis.set_major_formatter(INR_CR)
ax.set_title("Fraud loss concentration by ticket size (≥₹25K only)", loc="left", fontweight="600")
style_axes(ax, grid_axis="x")
save_chart(fig, "fraud_loss_by_amount_bucket", CHART_SUBDIR)

`Only ₹25–50K and ₹50K+ carry fraud loss (100% rate there by generator design). The footnote captures the 386K sub-₹25K txns that are noise on a full bucket chart.`

### Chart - Fraud rate: hour × day of week

In [ ]:
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
df["hour"] = df["transaction_ts"].dt.hour
df["dow"] = df["transaction_ts"].dt.day_name()
heat = df.pivot_table(index="dow", columns="hour", values="is_fraud", aggfunc="mean").mul(100)
heat = heat.reindex(day_order)
platform = df["is_fraud"].mean() * 100
vmin, vmax = platform - 0.5, heat.max().max() + 0.2

fig, ax = new_fig(11, 3.8)
minimal_heatmap(heat, ax, vmin=vmin, vmax=vmax, cbar_label="Fraud %")
ax.set_title(f"Fraud rate by hour & day (platform avg {platform:.2f}%)", loc="left", fontweight="600")
ax.set_xlabel("Hour (IST)")
ax.set_ylabel("")
save_chart(fig, "fraud_heatmap_hour_dow", CHART_SUBDIR)

`Peak cells are **weekday 2–5 AM IST** (~5–5.6% vs ~3.5% avg) — aligns with odd-hour scoring in dbt, not weekend late-night.`